# 11 — Embeddings: de texto a vectores

**Level 1 — LLM Engineering**

Convertimos frases en vectores numéricos (`nomic-embed-text` via Ollama) y medimos **similitud semántica** con coseno: frases que significan lo mismo quedan cerca; temas distintos quedan lejos.

In [1]:
import numpy as np
import requests

OLLAMA_HOST = "http://localhost:11434"
MODELO_EMBEDDING = "nomic-embed-text"


def obtener_embedding(texto: str) -> list[float]:
    """Devuelve el vector (embedding) de un texto usando Ollama."""
    url = f"{OLLAMA_HOST}/api/embeddings"
    payload = {"model": MODELO_EMBEDDING, "prompt": texto}
    response = requests.post(url, json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["embedding"]


def similitud_coseno(a: list[float], b: list[float]) -> float:
    """Mide la similitud entre dos vectores: 1 = iguales, 0 = sin relacion."""
    va = np.array(a)
    vb = np.array(b)
    return float(np.dot(va, vb) / (np.linalg.norm(va) * np.linalg.norm(vb)))

## Generar embeddings

In [2]:
frases = [
    "El gato duerme en el sofa",
    "Un gato descansa sobre el sofa",
    "Hoy hace mucho calor en la ciudad",
    "El clima esta muy caluroso hoy",
]

print(f"=== EMBEDDINGS con {MODELO_EMBEDDING} ===")
vectores = {}
for frase in frases:
    vectores[frase] = obtener_embedding(frase)
    print(f"[{len(vectores[frase])} dimensiones] {frase}")

=== EMBEDDINGS con nomic-embed-text ===


[768 dimensiones] El gato duerme en el sofa
[768 dimensiones] Un gato descansa sobre el sofa
[768 dimensiones] Hoy hace mucho calor en la ciudad
[768 dimensiones] El clima esta muy caluroso hoy


## Similitud coseno entre pares

In [3]:
print("=== SIMILITUD COSENO entre pares ===")
pares = [
    (frases[0], frases[1]),
    (frases[0], frases[2]),
    (frases[2], frases[3]),
    (frases[0], frases[3]),
]
for a, b in pares:
    sim = similitud_coseno(vectores[a], vectores[b])
    print(f"{sim:.3f} | {a}")
    print(f"        | {b}")
    print()

=== SIMILITUD COSENO entre pares ===
0.919 | El gato duerme en el sofa
        | Un gato descansa sobre el sofa

0.526 | El gato duerme en el sofa
        | Hoy hace mucho calor en la ciudad

0.689 | Hoy hace mucho calor en la ciudad
        | El clima esta muy caluroso hoy

0.617 | El gato duerme en el sofa
        | El clima esta muy caluroso hoy



## Conclusión

- Frases sobre el mismo tema (gatos / clima) → similitud alta (~0.9)
- Temas distintos → similitud baja (~0.5-0.6)

Eso es lo que permite a ChromaDB (siguiente section) buscar "lo que significa algo parecido", no solo coincidencia de palabras.